In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split , GridSearchCV, TimeSeriesSplit,RandomizedSearchCV,cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

In [3]:
df = pd.read_csv('train.csv')
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)
df.drop(['Store', 'Dept'],axis=1,inplace=True)
df = df.groupby(df.index).agg({'Weekly_Sales': 'sum', 'IsHoliday': 'max'})
df.rename(columns={'Weekly_Sales': 'all_store_Weekly_Sales', 'IsHoliday': 'IsHoliday'}, inplace=True)
df.head()

,all_store_Weekly_Sales,IsHoliday
Date,,
2010-02-05,49750740.50,False
2010-02-12,48336677.63,True
2010-02-19,48276993.78,False
2010-02-26,43968571.13,False
2010-03-05,46871470.30,False


In [4]:
df['lag1'] = df['all_store_Weekly_Sales'].shift(1)

df['lag2'] = df['all_store_Weekly_Sales'].shift(2)

df['lag3'] = df['all_store_Weekly_Sales'].shift(3)

df['lag4'] = df['all_store_Weekly_Sales'].shift(4)

df['lag5'] = df['all_store_Weekly_Sales'].shift(5)

df['lag6'] = df['all_store_Weekly_Sales'].shift(6)

df['lag7'] = df['all_store_Weekly_Sales'].shift(7)

df['month'] = df.index.month

df['week'] = (df.index.day - 1) // 7 + 1

df['scales_change'] = df['all_store_Weekly_Sales'].pct_change().shift(1) # velocity in sales growth

df['Pre_Holiday_Week'] =  df['IsHoliday'].shift(-1).astype(bool)

df['moving_avg'] = df['all_store_Weekly_Sales'].rolling(window=4).mean()

df['moving_median'] = df['all_store_Weekly_Sales'].rolling(window=4).median()

df['moving_std'] = df['all_store_Weekly_Sales'].rolling(window=4).std()

df['moving_max'] = df['all_store_Weekly_Sales'].rolling(window=4).max()

df['moving_min'] = df['all_store_Weekly_Sales'].shift(1).rolling(window=4).mean()

df['moving_sum'] = df['all_store_Weekly_Sales'].rolling(window=4).sum()

In [5]:
df.columns

Index(['all_store_Weekly_Sales', 'IsHoliday', 'lag1', 'lag2', 'lag3', 'lag4',
       'lag5', 'lag6', 'lag7', 'month', 'week', 'scales_change',
       'Pre_Holiday_Week', 'moving_avg', 'moving_median', 'moving_std',
       'moving_max', 'moving_min', 'moving_sum'],
      dtype='object')

In [6]:
df.dropna(inplace=True)

df.head()

,all_store_Weekly_Sales,IsHoliday,lag1,lag2,lag3,lag4,lag5,lag6,lag7,month,week,scales_change,Pre_Holiday_Week,moving_avg,moving_median,moving_std,moving_max,moving_min,moving_sum
Date,,,,,,,,,,,,,,,,,,,
2010-03-26,44133961.05,False,44988974.64,45925396.51,46871470.30,43968571.13,48276993.78,48336677.63,49750740.50,3,4,-0.020390,False,4.547995e+07,4.545719e+07,1.181453e+06,46871470.30,4.543860e+07,1.819198e+08
2010-04-02,50423831.26,False,44133961.05,44988974.64,45925396.51,46871470.30,43968571.13,48276993.78,48336677.63,4,1,-0.019005,False,4.636804e+07,4.545719e+07,2.801089e+06,50423831.26,4.547995e+07,1.854722e+08
2010-04-09,47365290.44,False,50423831.26,44133961.05,44988974.64,45925396.51,46871470.30,43968571.13,48276993.78,4,2,0.142518,False,4.672801e+07,4.617713e+07,2.817715e+06,50423831.26,4.636804e+07,1.869121e+08
2010-04-16,45183667.08,False,47365290.44,50423831.26,44133961.05,44988974.64,45925396.51,46871470.30,43968571.13,4,3,-0.060657,False,4.677669e+07,4.627448e+07,2.779078e+06,50423831.26,4.672801e+07,1.871067e+08
2010-04-23,44734452.56,False,45183667.08,47365290.44,50423831.26,44133961.05,44988974.64,45925396.51,46871470.30,4,4,-0.046060,False,4.692681e+07,4.627448e+07,2.599128e+06,50423831.26,4.677669e+07,1.877072e+08


In [7]:
df.shape

(136, 19)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 136 entries, 2010-03-26 to 2012-10-26
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   all_store_Weekly_Sales  136 non-null    float64
 1   IsHoliday               136 non-null    bool   
 2   lag1                    136 non-null    float64
 3   lag2                    136 non-null    float64
 4   lag3                    136 non-null    float64
 5   lag4                    136 non-null    float64
 6   lag5                    136 non-null    float64
 7   lag6                    136 non-null    float64
 8   lag7                    136 non-null    float64
 9   month                   136 non-null    int32  
 10  week                    136 non-null    int32  
 11  scales_change           136 non-null    float64
 12  Pre_Holiday_Week        136 non-null    bool   
 13  moving_avg              136 non-null    float64
 14  moving_median          

In [9]:
x = df.drop('all_store_Weekly_Sales',axis=1)
y = df['all_store_Weekly_Sales']

x_train = x[:-12]
x_test = x[-12:]

y_train = y[:-12]
y_test = y[-12:]

In [10]:
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(124, 18)
(12, 18)
(124,)
(12,)


In [11]:
par_xgb ={
    'learning_rate': [0.2,0.1],
    'max_depth': [4,3],
    'n_estimators': [500, 400],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 2],
    'reg_lambda': [1, 10]
}

xgb = XGBRegressor(objective='reg:squarederror', random_state=42)
grid_xgb = GridSearchCV(xgb, par_xgb, cv=5, n_jobs=-1, verbose=1,scoring='neg_root_mean_squared_error')
grid_xgb.fit(x_train, y_train)

Fitting 5 folds for each of 128 candidates, totalling 640 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'colsample_bytree': [0.8, 1.0], 'learning_rate': [0.2, 0.1], 'max_depth': [4, 3], 'min_child_weight': [1, 2], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each

In [12]:
print(grid_xgb.best_params_)
print(grid_xgb.best_score_)

{'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 4, 'min_child_weight': 2, 'n_estimators': 500, 'reg_lambda': 10, 'subsample': 0.8}
-2221513.391259258


In [13]:

feat_imp = pd.Series(grid_xgb.best_estimator_.feature_importances_, index=x_train.columns)

print(feat_imp.sort_values(ascending=False))

best_feature = feat_imp[feat_imp > 0.02].index

print(best_feature)


month               0.465194
moving_avg          0.105342
moving_sum          0.066958
week                0.054094
lag5                0.052725
moving_std          0.047812
IsHoliday           0.046547
lag1                0.042632
moving_median       0.023026
moving_max          0.022509
lag7                0.016375
scales_change       0.013160
lag3                0.010222
moving_min          0.009593
lag4                0.009064
lag2                0.008185
lag6                0.005926
Pre_Holiday_Week    0.000636
dtype: float32
Index(['IsHoliday', 'lag1', 'lag5', 'month', 'week', 'moving_avg',
       'moving_median', 'moving_std', 'moving_max', 'moving_sum'],
      dtype='object')


In [14]:
y_pred = grid_xgb.predict(x_test)                               

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print('RMSE:', rmse)
print('R2:', r2)

RMSE: 929117.2578325291
R2: 0.6111154840226525


In [15]:
x  = df[best_feature]
y = df['all_store_Weekly_Sales']

x_train = x[:-12]
x_test = x[-12:]

y_train = y[:-12]
y_test = y[-12:]


In [17]:
import optuna

c:\Users\kumar\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
def optimise_xgb(trial):
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    max_depth = trial.suggest_int('max_depth', 3, 9)
    n_estimators = trial.suggest_int('n_estimators', 100, 800)
    subsample = trial.suggest_float('subsample', 0.5, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.2, 1.0)
    colsample_bylevel = trial.suggest_float('colsample_bylevel', 0.2, 1.0)
    min_child_weight = trial.suggest_int('min_child_weight', 2, 9)
    gamma = trial.suggest_float('gamma', 0, 9)
    reg_lambda = trial.suggest_float('reg_lambda', 0, 9)
    reg_alpha = trial.suggest_float('reg_alpha', 0, 9)


    model = XGBRegressor(
        learning_rate=learning_rate,
        max_depth=max_depth,
        n_estimators=n_estimators,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        colsample_bylevel=colsample_bylevel,
        min_child_weight=min_child_weight,
        gamma=gamma,
        reg_lambda=reg_lambda,
        reg_alpha=reg_alpha,
        random_state=42
    )

    model.fit(x_train, y_train)

    tscv = TimeSeriesSplit(n_splits=5)

    score = cross_val_score(model, x_train, y_train, cv=tscv, scoring='neg_root_mean_squared_error').mean()

    return score

In [19]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())
study.optimize(optimise_xgb, n_trials=30)


[I 2026-05-07 02:44:22,512] A new study created in memory with name: no-name-30fc1d06-76f1-4e3c-b841-f78ce17c6213
[I 2026-05-07 02:44:24,710] Trial 0 finished with value: -5301552.876618385 and parameters: {'learning_rate': 0.14734827442950585, 'max_depth': 8, 'n_estimators': 568, 'subsample': 0.7698686102592363, 'colsample_bytree': 0.23338827742497453, 'colsample_bylevel': 0.47755247483849816, 'min_child_weight': 9, 'gamma': 3.0269410350975967, 'reg_lambda': 5.453695400742728, 'reg_alpha': 3.2172822283110944}. Best is trial 0 with value: -5301552.876618385.
[I 2026-05-07 02:44:26,202] Trial 1 finished with value: -3905727.8252353496 and parameters: {'learning_rate': 0.0993081910056582, 'max_depth': 4, 'n_estimators': 599, 'subsample': 0.8954474027203232, 'colsample_bytree': 0.6883506466354494, 'colsample_bylevel': 0.7858537312350473, 'min_child_weight': 5, 'gamma': 5.36401307573799, 'reg_lambda': 4.677029776985244, 'reg_alpha': 1.8240605304157267}. Best is trial 1 with value: -3905727

In [20]:
print(study.best_params)
print(study.best_value)

{'learning_rate': 0.0993081910056582, 'max_depth': 4, 'n_estimators': 599, 'subsample': 0.8954474027203232, 'colsample_bytree': 0.6883506466354494, 'colsample_bylevel': 0.7858537312350473, 'min_child_weight': 5, 'gamma': 5.36401307573799, 'reg_lambda': 4.677029776985244, 'reg_alpha': 1.8240605304157267}
-3905727.8252353496


In [21]:

best_params = study.best_params

best_model = XGBRegressor(**best_params, random_state=42)

best_model.fit(x_train, y_train)

y_pred = best_model.predict(x_test)                               

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print('RMSE:', rmse)
print('R2:', r2)

RMSE: 1176890.1850741915
R2: 0.3760475713402692


In [44]:
def optimise_xgb(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.9),
        'max_depth': trial.suggest_int('max_depth', 2, 9),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'subsample': trial.suggest_float('subsample', 0.2, 1.0),
        'gamma': trial.suggest_int('gamma', 2, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 2, 9),
        'reg_lambda': trial.suggest_int('reg_lambda', 0, 9), 
        'reg_alpha': trial.suggest_int('reg_alpha', 0, 9)
    }

    model = XGBRegressor(**params, random_state=42)
    
    tscv = TimeSeriesSplit(n_splits=5)
    
    score = cross_val_score(model, x_train, y_train, cv=tscv, scoring='neg_root_mean_squared_error').mean()
    return score

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())
study.optimize(optimise_xgb, n_trials=25)


[I 2026-05-07 02:51:04,667] A new study created in memory with name: no-name-e0350b08-19f7-41c9-b127-9b9b32192a8b
[I 2026-05-07 02:51:06,156] Trial 0 finished with value: -5805929.235589166 and parameters: {'learning_rate': 0.23572685176684238, 'max_depth': 6, 'n_estimators': 447, 'subsample': 0.33015348027815705, 'gamma': 3, 'min_child_weight': 8, 'reg_lambda': 5, 'reg_alpha': 9}. Best is trial 0 with value: -5805929.235589166.
[I 2026-05-07 02:51:08,156] Trial 1 finished with value: -4506117.305321024 and parameters: {'learning_rate': 0.010687166614376154, 'max_depth': 6, 'n_estimators': 957, 'subsample': 0.5744012275570651, 'gamma': 4, 'min_child_weight': 7, 'reg_lambda': 0, 'reg_alpha': 3}. Best is trial 1 with value: -4506117.305321024.
[I 2026-05-07 02:51:09,304] Trial 2 finished with value: -4567430.06560903 and parameters: {'learning_rate': 0.08232267311666025, 'max_depth': 9, 'n_estimators': 426, 'subsample': 0.7407304818101972, 'gamma': 5, 'min_child_weight': 3, 'reg_lambda':

In [45]:
print(study.best_params)
print(study.best_value)

{'learning_rate': 0.5289387683777043, 'max_depth': 9, 'n_estimators': 129, 'subsample': 0.9894372630500964, 'gamma': 2, 'min_child_weight': 4, 'reg_lambda': 7, 'reg_alpha': 5}
-3876058.7038208223


In [46]:
best_params = study.best_params

best_model = XGBRegressor(**best_params, random_state=42)

best_model.fit(x_train, y_train)

y_pred = best_model.predict(x_test)                               

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print('RMSE:', rmse)
print('R2:', r2)

RMSE: 680108.049436037
R2: 0.7916298868756623


In [47]:
print(study.best_params)

{'learning_rate': 0.5289387683777043, 'max_depth': 9, 'n_estimators': 129, 'subsample': 0.9894372630500964, 'gamma': 2, 'min_child_weight': 4, 'reg_lambda': 7, 'reg_alpha': 5}


{'learning_rate': 0.5289387683777043, 'max_depth': 9, 'n_estimators': 129, 'subsample': 0.9894372630500964, 'gamma': 2, 'min_child_weight': 4, 'reg_lambda': 7, 'reg_alpha': 5}

In [49]:
import joblib   

joblib.dump(best_model, 'xgb_model.pkl')

['xgb_model.pkl']